In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("RecoScale-ALS-Train") \
    .master("local[*]") \
    .config("spark.driver.memory", "8g") \
    .config("spark.driver.memoryOverhead", "2g") \
    .config("spark.driver.maxResultSize", "2g") \
    .config("spark.memory.fraction", "0.7") \
    .config("spark.memory.storageFraction", "0.2") \
    .config("spark.sql.shuffle.partitions", "200") \
    .config("spark.local.dir", "/mnt/d/spark_tmp") \
    .config("spark.executor.heartbeatInterval", "60s") \
    .config("spark.network.timeout", "300s") \
    .config("spark.serializer", "org.apache.spark.serializer.KryoSerializer") \
    .config("spark.kryoserializer.buffer.max", "512m") \
    .getOrCreate()
print('Spark Started...')

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/04/13 09:00:07 WARN Utils: Your hostname, JACK, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/04/13 09:00:07 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/13 09:00:09 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/04/13 09:00:09 WARN SparkConf: Note that spark.local.dir will be overridden by the value set by the cluster manager (via SPARK_LOCAL_DIRS in standalone/kubernetes and LOCAL_DIRS in YARN).
26/04/13 09:00:10 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
26/04/13 09:00:10 WARN Utils: Service 'SparkUI

Spark Started...


In [2]:
df = spark.read.parquet("/mnt/d/Projects/recoscale/data/df_indexed_110M.parquet")

In [3]:
from pyspark.sql import functions as F
print("\n=== Reviews per User (distribution) ===")
user_activity = df.groupBy("user_id").count().withColumnRenamed("count", "review_count")
user_activity.select(
    F.min("review_count").alias("min"),
    F.percentile_approx("review_count", 0.25).alias("p25"),
    F.percentile_approx("review_count", 0.50).alias("median"),
    F.percentile_approx("review_count", 0.75).alias("p75"),
    F.max("review_count").alias("max"),
    F.avg("review_count").alias("mean")
).show()


=== Reviews per User (distribution) ===


+---+---+------+---+----+-----------------+
|min|p25|median|p75| max|             mean|
+---+---+------+---+----+-----------------+
|  1|  1|     2|  4|4868|3.505633332514759|
+---+---+------+---+----+-----------------+



In [4]:
user_activity = df.groupBy("user_id").count().withColumnRenamed("count", "review_count").show()

+--------------------+------------+
|             user_id|review_count|
+--------------------+------------+
|AGEID42W2PWAV3ERO...|           2|
|AFXABTTWQTMQTCFES...|          21|
|AHR3NCI3S2A6AOOF6...|          30|
|AHOW6WZXTZDNMGPZO...|          15|
|AGVIQUGHFW3MVDCQV...|          27|
|AH7CGGT4BWTND2ZHS...|          12|
|AHZM5QZYZU6IU5TYK...|           5|
|AF6VBSOI3NJNQFRLS...|           2|
|AFSBG6AADJIF2TDR5...|          48|
|AHUJOJVPUGWFRCAR5...|           1|
|AEDZASP52Q66QJPY4...|           3|
|AGPHPERJM45OM3D4F...|          18|
|AFKRXHUCASDFNKCAB...|          64|
|AE4JC727REGYA52XI...|           4|
|AENF76NJSN64O3YGE...|          29|
|AFIYKGBDCZO2QZWYL...|           4|
|AFVP5LGA4POREN22R...|           4|
|AGKE3HUWJ5N44TJIK...|          13|
|AHLSUO7H5XLHMXK2B...|           6|
|AHCA3JGZIEJ4SKULW...|           1|
+--------------------+------------+
only showing top 20 rows
